# Chapter 04 제출 답안 양식. pandas로 데이터에 질문하기

## 0. 제출 정보
- 이름: 양성용
- GitHub ID: dydtlsrl@gmail.com
- 작성일: 2026-09-10
- 최종 제출 URL: https://github.com/dydtlsrl/ai-data-analysis/blob/main/00_llm-data-analysis-course/notebooks/chapter04/chapter04.ipynb

## 1. 질문과 필요한 데이터 선택
### 내가 확인하려는 질문
완료된 주문을 기준으로 월별 매출 금액과 주문 수가 어떻게 달라지는지 확인하고 싶었다.  
또한 어떤 카테고리, 상품, 고객에서 금액이 많이 발생했는지도 함께 확인하고 싶었다.  
### 사용한 파일/컬럼  
- orders.csv: 
    - order_id, customer_id, order_date, order_status를 사용했다.   
    - 주문 번호로 주문 상세를 연결하고, 완료 여부와 주문 날짜를 확인하기 위해서다.  
- order_items.csv:  
    - order_id, product_id, quantity, unit_price를 사용했다.  
    - 수량과 단가를 곱해 주문 상품 한 줄의 금액(line_total)을 계산하기 위해서다.  
- products.csv:  
    - product_id, product_name, category를 사용했다.   
    - 상품별·카테고리별로 금액을 나누어 보기 위해서다.  
- customers.csv: 
    - customer_id, city를 사용했다. 
    - 고객별 구매 금액을 만들고 고객이 사는 도시 정보까지 확인하기 위해서다.  
### 결과 관찰
한 파일에 모든 정보가 들어 있지 않았다. order_items에는 수량과 단가가 있지만 주문 상태와  
날짜가 없고, orders에는 주문 상태와 날짜가 있지만 상품별 수량과 단가가 없다.   
그래서 order_id를 기준으로 두 파일을 연결해야 했다.  
### 나의 해석과 판단
이번 분석의 기준은 실제로 완료된 주문이다. 따라서 orders의 order_status가   
completed인 행만 남긴 뒤 금액을 집계했다. 주문 상태를 확인하지 않고 모든 주문을  
더하면 취소되었거나 환불된 주문까지 포함될 수 있어 결과가 달라질 수 있다.  
### 업무·분석적 의미
질문에 맞는 파일과 컬럼을 먼저 정하면 필요한 데이터만 연결할 수 있다. 이 결과는   
많이 팔린 카테고리, 상품, 기간, 고객을 찾아 재고·판매 전략을 검토하는 기초 자료로 사용할 수 있다.  
### 한계와 추가 확인 사항
현재 데이터에는 할인, 쿠폰, 배송비, 세금, 실제 환불 금액 정보가 없으므로 계산한   
total_sales는 회계상 순매출과 다를 수 있다. 또한 원본 데이터에 주문 정보나  
상품 정보가 빠진 행이 있는지도 병합 과정에서 따로 확인해야 한다.  

## 2. 필터링·정렬·파생 컬럼
- 적용한 필터 조건: 주문 상태가 completed인 주문만 남겼다. 
cancelled나 refunded처럼 완료되지 않은 주문은 이번 매출 분석에서 제외했다.

- 정렬 기준: 카테고리별·상품별 결과는 total_sales가 큰 순서(내림차순)로 정렬했다.  
월별 결과는 order_month가 빠른 달부터 보이도록 오름차순으로 정렬했다.

- 만든 파생 컬럼: line_total과 order_month를 만들었다. line_total은 주문 상품 한 줄의  
금액이고, order_month는 주문일을 YYYY-MM 형태의 월 단위로 바꾼 컬럼이다.

- `line_total` 계산식: quantity × unit_price. 예를 들어 수량이 2개이고  
단가가 10,000원이면 line_total은 20,000원이다.

![필터와 파생 컬럼](images/step02_transform.png)
### 결과 관찰
완료 주문만 남긴 뒤 각 상품 행의 금액을 line_total로 계산했다.  
이 값을 기준으로 카테고리별, 상품별, 월별, 고객별 매출을 집계할 수 있었다.
### 나의 해석과 판단
이번에는 실제로 완료된 주문만 보려고 completed 조건을 사용했다.  
만약 cancelled나 refunded 주문까지 포함하면 실제 완료 기준의 매출보다 금액이  
커지거나, 취소·환불 주문이 섞여 결과를 잘못 해석할 수 있다. 그래서 먼저 어떤 주문  
상태를 분석에 포함할지 정하는 것이 중요하다고 생각했다.
### 업무·분석적 의미
line_total을 먼저 만들면 행 단위 금액이 명확해져서 이후 집계 기준이 단순해진다.  
또한 매출이 큰 카테고리나 상품을 바로 확인할 수 있어 판매 현황을 빠르게 파악하는 데 도움이 된다.
### 한계와 추가 확인 사항
이번 line_total은 수량과 단가만 곱한 값이다. 할인, 쿠폰, 배송비, 세금, 환불 금액은 
반영하지 않았으므로 이 결과를 회계상 순매출이라고 단정하면 안 된다. 실제 업무에서는  
이런 금액이 별도 컬럼에 있는지 추가로 확인해야 한다.

## 3. merge 검증
- 병합한 데이터: 먼저 order_items에 orders의 주문 정보(customer_id, order_date, order_status)를 붙였다.  
그다음 completed 주문 상세에 products의 상품 정보(product_name, category, price)를 붙였다.
- 사용한 key: 주문 정보 병합은 order_id를 기준으로 했고, 상품 정보 병합은 product_id를 기준으로 했다.
- `validate` 결과: orders는 order_id가 중복되지 않아 many_to_one 조건을 통과했다. 즉 주문 상세는  
한 주문에 여러 행이 있을 수 있고, 주문 정보는 주문 번호마다 한 행이어야 한다는 관계를 확인했다.  
products 병합도 상품 정보가 product_id마다 한 행이라는 many_to_one 조건으로 확인했다.
- `indicator` 결과: orders 병합은 both 765건, left_only 1건(order_id 320), right_only 0건이었다. 
 products 병합은 completed 주문 475행 중 both 474건, left_only 1건(product_id 108, 주문 번호 299), 
  right_only 0건이었다.
- 병합 전/후 행 수: orders 병합은 766행에서 766행으로 유지됐다. products 병합은 completed  
주문 상세 475행에서 475행으로 유지됐다.
![merge 검증](images/step03_merge.png)
### 결과 관찰
두 병합 모두 전후 행 수가 같아서 병합 때문에 주문 상세 행이 갑자기 늘어나거나 사라지지 않았음을 확인했다.  
다만 order_id 320은 orders 원본에 없었고, product_id 108은 products 원본에 없어서 left_only가 각각 1건씩 나왔다.
### 나의 해석과 판단
orders에 없는 order_id 320은 완료 주문인지 판단할 수 없으므로 completed 분석에서 제외했다.  
반면 주문 번호 299는 완료 주문이지만 상품 정보만 없는 경우였다. 이 행의 금액은 유지하고 상품명은 
상품 정보 없음, 카테고리는 미분류로 처리했다. 이렇게 해야 매출 700,000원이 집계에서 빠지지 않는다.
### 업무·분석적 의미
merge 전에 키의 중복과 행 수를 확인하면 데이터가 중복 연결되어 매출이 부풀려지는 문제를 막을 수 있다.  
indicator를 보면 어느 쪽 원본 데이터가 비어 있는지도 바로 찾을 수 있다.
### 한계와 추가 확인 사항
현재 미매칭 2건은 분석 규칙에 따라 처리했지만, 실제 업무에서는 왜 주문 정보 또는 상품 마스터 정보가  
빠졌는지 원본 시스템 담당자에게 확인해야 한다. 특히 미분류 상품은 카테고리별 분석 결과에 영향을 줄 수 있다.

## 4. completed 주문 범위와 집계
- 분석 범위 정의: orders와 연결할 수 있고 주문 상태가 completed인 주문만 분석했다.  
    주문 정보가 없는 order_id 320은 완료 여부를 알 수 없어 제외했다.
- 카테고리별 결과: 8개 카테고리로 집계했다. 스포츠가 31,743,000원으로 가장 컸고, 
    상품 정보가 없는 1건은 미분류 700,000원으로 따로 표시했다.
- 상품별 결과: 99개 상품 기준으로 집계했다. 스포츠 상품 041이 총 5,705,000원으로 가장 높았다.  
- 월별 결과: 13개월로 집계했다. 2025-11이 주문 25건, total_sales 23,611,000원으로 가장 높았다.  
- 고객별 결과: 100명 기준으로 집계했다. customer_id 117(성남)이 주문 5건,  
    total_sales 4,100,000원으로 가장 높았다.
![핵심 집계 결과](images/step04_groupby.png)
### 결과 관찰
스포츠 카테고리가 총매출 31,743,000원으로 가장 높았고, 상위 상품도 스포츠 상품 041이었다.  
월별로는 2025년 11월이 가장 높았다. 따라서 이번 데이터에서는 스포츠 카테고리와  
2025년 11월의 주문 흐름을 먼저 자세히 보는 것이 좋다고 판단했다.
### 나의 해석과 판단
카테고리·상품·월별 결과를 같이 봐야 한쪽 결과만 보고 판단하는 실수를 줄일 수 있다고 생각했다. 
 예를 들어 스포츠가 전체적으로 높아도, 특정 한 상품이나 특정 한 달에만 매출이 몰렸는지는 추가로 확인해야 한다.
### 업무·분석적 의미
스포츠 상품의 재고 확보, 관련 상품 추천, 11월 판매 증가 원인 확인으로 이어갈 수 있다.   
고객별 집계는 구매 금액이 큰 고객의 구매 패턴이나 지역별 차이를 보는 출발점으로 사용할 수 있다.
### 한계와 추가 확인 사항
total_sales는 수량과 단가를 곱한 주문 상세 금액의 합계다. 할인, 쿠폰, 배송비, 세금,  
실제 환불 금액이 반영되지 않았으므로 회계상 순매출과 같다고 단정할 수 없다. 또한 미분류  
700,000원은 상품 정보가 누락된 결과이므로 원본 상품 마스터를 추가 확인해야 한다.


## 5. 총합 일치 검증
- 원본 completed `line_total` 합계: 149,690,000원
- 카테고리 합계: 149,690,000원
- 월별 합계: 149,690,000원
- 고객별 합계: 149,690,000원
- 차이 여부: 차이 없음. 네 집계의 합계가 모두 같아서 PASS로 확인했다.
![총합 검증](images/step05_total_check.png)
### 나의 해석과 판단
집계표마다 총합이 같아야 같은 completed 주문 데이터를 서로 다른 기준으로만 나누어 본 것이라고 판단할 수 있다. 만약 합계가 다르면 먼저 completed 필터가 모든 집계에 동일하게 적용됐는지, 미매칭 행을 제외하거나 중복으로 넣지 않았는지, groupby에 사용한 컬럼이 맞는지 순서대로 확인해야 한다.

## 6. LLM pandas 코드 검증
- LLM Prompt 요약: orders와 order_items로 completed 주문 기준 월별 금액을 계산하되, 
    line_total 계산, 키 고유성, many-to-one 병합, indicator, 행 수, 미매칭, 날짜 변환,  
    completed 필터, 월별 매출·고유 주문 수를 모두 확인하도록 요청했다.
- 제안 코드 요약: line_total을 만든 뒤 orders.order_id 고유성을 확인하고, left merge와  
    validate=many_to_one, indicator를 사용했다. 이후 completed만 필터링하고 order_month  
    기준으로 total_sales와 고유 order_count를 집계하는 방식이었다.
- 실제 컬럼/범위와 맞지 않은 부분: 날짜 변환 실패는 병합 후에 확인하면 order_id 320처럼 주문  
    정보가 없는 행의 NaN까지 날짜 오류처럼 보일 수 있다. 날짜 원본의 변환 실패는 orders에서  
    병합 전에 확인해야 하며, 실제 원본 날짜 변환 실패는 0건이었다. 또한 product_id 108 미매칭  
    1건을 그냥 버리면 700,000원이 빠질 수 있다.
- 수정한 내용: 주문 정보 미매칭(order_id 320)은 completed 여부를 판단할 수 없어 제외했다. 
    상품 정보 미매칭(product_id 108)은 금액을 유지하고 상품명은 상품 정보 없음, 카테고리는  
    미분류로 처리했다. 날짜 검증 위치와 미매칭 처리 기준을 실제 데이터에 맞게 보완했다.
- 최종 판단: 수정 후 사용. LLM 코드는 초안으로 유용하지만, 실제 컬럼명·병합 관계·미매칭  
    처리·결과 의미는 직접 검증한 뒤에만 사용해야 한다.  
![LLM 코드 검증](images/step06_llm_validation.png)

### 나의 해석과 판단
코드가 실행된다는 것은 문법 오류가 없다는 뜻일 뿐이다. 잘못된 키로 병합하거나 완료되지 않은  
주문을 포함해도 실행은 될 수 있다. 그래서 실행 결과뿐 아니라 행 수, indicator, 미매칭,  
총합까지 확인해야 분석 결과를 믿을 수 있다.

## 7. Chapter 04 최종 인사이트
### 가장 의미 있다고 생각한 결과 2가지
1. 완료 주문 기준으로 스포츠 카테고리의 total_sales가 31,743,000원으로 가장 높았다.  
상위 상품도 스포츠 상품 041(5,705,000원)이어서 스포츠 상품군을 우선 분석할 근거가 생겼다.  
2. 월별로는 2025년 11월의 total_sales가 23,611,000원, 주문 수가 25건으로 가장 높았다.  
 이 달에 프로모션, 계절 요인, 특정 상품 판매 증가가 있었는지 추가로 확인할 필요가 있다.  
### 그 결과를 뒷받침하는 수치/표
- 전체 completed 주문 상세의 line_total 합계: 149,690,000원
- 카테고리별 상위: 스포츠 31,743,000원
- 상품별 상위: 스포츠 상품 041, 5,705,000원
- 월별 상위: 2025-11, 주문 25건, 23,611,000원
- 고객별 상위: customer_id 117(성남), 주문 5건, 4,100,000원
### 추가로 확인하고 싶은 질문
- 2025년 11월 매출이 가장 높았던 이유는 무엇인가? 할인 행사, 계절성, 특정 상품 판매 증가 중 어떤 요인이 컸는가?  
- 스포츠 카테고리의 높은 매출이 여러 상품에서 고르게 나온 것인가, 일부 인기 상품에 집중된 것인가?  
- 미분류 700,000원에 해당하는 product_id 108의 상품 정보는 왜 products 원본에 없는가?  
### 현재 결과의 한계
- 할인, 쿠폰, 배송비, 세금, 실제 환불 금액이 없어 total_sales를 회계상 순매출로 볼 수 없다.  
- orders에 없는 order_id 320은 completed 여부를 판단할 수 없어 분석에서 제외했다.  
- 상품 정보가 없는 product_id 108은 미분류로 유지했으므로 카테고리별 결과에 700,000원이 미분류로 포함되어 있다.  
## 최종 제출 체크
- [x] 핵심 셀 Output이 남아 있습니다.  
- [x] merge와 총합 검증 Evidence가 있습니다.  
- [x] 결과 관찰과 해석이 구분되어 있습니다.  
- [x] LLM 코드를 검증했습니다.  
- [x] 개인정보/Secret이 없는지 제출 전 마지막으로 확인합니다.  
- [x] `chapter04/chapter04.ipynb`가 GitHub에서 정상 표시되는지 확인합니다.  
- [x] 최종 Notebook 파일 URL을 제출합니다.  